# 00 - Setup and data load

**Purpose**: verify the environment is healthy, load every dataframe the
other notebooks need, and write them to a parquet cache so the
downstream notebooks open fast.

Run this notebook **first**. After the final cell finishes, open any of
the 01-06 notebooks in order -- they all read from the cache produced
here and never touch the SQLite database directly.


## 1. Environment health check

Runs ``endpoint_ck_analysis.doctor.doctor()`` which prints a table of required
pieces (Python version, packages, database file, writable cache,
region-prior source). Everything should be ``[OK]``.

If anything is ``[FAIL]``:
  - Missing package? The "detail" column gives the ``pip install`` command.
  - Database file not found? Copy ``connectome.db`` into
    ``endpoint_ck_analysis/_bundled_data/`` (the folder you see next to this
    notebook's parent) or set ``ENDPOINT_CK_ANALYSIS_DB`` to its location.
  - Still stuck? Open ``TROUBLESHOOTING.md`` in the project root.


In [ ]:
from endpoint_ck_analysis.doctor import doctor # entry point that runs every diagnostic check and prints a status table
doctor() # prints the [OK]/[FAIL]/[INFO] table; returns True when there are no blockers


## 2. Load all dataframes

``load_all`` reads ``connectome.db`` (location resolved by
``config.get_db_path()``), runs every filter/aggregation/pivot from
Section 6 of the original notebook, and returns a :class:`LoadedData`
container with everything downstream code needs.

On first run this takes ~30 seconds (SQL + aggregation). Subsequent
runs read the cache in under a second unless you delete it or pass
``use_cache=False`` to force a rebuild.


In [ ]:
from endpoint_ck_analysis.data_loader import load_all # one-shot loader that returns a LoadedData with every dataframe downstream notebooks consume

data = load_all(use_cache=False, write_cache=True) # use_cache=False forces a fresh DB rebuild; write_cache=True saves parquet so notebooks 01-08 don't re-touch the DB. Set use_cache=True after the first run for fast reloads.

print()
print('Base dataframes:') # AKDdf and friends are the per-reach / per-region tables; everything else is derived from these
for name in ['AKDdf', 'FKDdf', 'ACDUdf', 'ACDGdf', 'FCDUdf', 'FCDGdf']: # iterate by name string so the loop also prints the variable name alongside the shape
    print(f'  {name}: {getattr(data, name).shape}') # getattr(data, name) pulls the dataframe attribute matching the loop variable; .shape gives (rows, columns)

print()
print('Connectomics wide pivots:') # one row per subject, one column per region_hemi; produced by helpers.connectivity.pivot_connectivity
for name in ['ACDUdf_wide', 'ACDGdf_wide', 'FCDUdf_wide', 'FCDGdf_wide']:
    print(f'  {name}: {getattr(data, name).shape}')

print()
print(f'Matched subjects (both kinematics and connectomics): {list(data.matched_subjects)}') # the analyzable cohort: only subjects in this list participate in PLS or any cross-block analysis


## 3. Quick-look previews

Spot-check that the data resembles what you expect before running the
analytical notebooks. Uncomment any lines that would be useful to see.


In [ ]:
# data.AKDdf.head()
# data.FCDGdf_wide
# data.AKDdf_agg_contact.head()
pass


## Next

- **01_connectivity_pca.ipynb** -- PCA on the connectivity matrix.
- **02_kinematic_pca.ipynb** -- PCA on kinematics, one per phase, with sign alignment.
- **03_kinematic_clustering.ipynb** -- hierarchical clustering of kinematic features.
- **04_pls_variants.ipynb** -- PLS across connectivity and kinematics (injury / deficit / recovery).
- **05_lmm_phase_effects.ipynb** -- Linear mixed models for per-feature phase effects.
- **06_pellet_validation.ipynb** -- manual vs algorithmic pellet scoring agreement.
- **99_figure_gallery.ipynb** -- all saved figures in one place.


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the doctor + data load tells you (click to expand)</summary>

**Doctor table**: every required check should show `[OK]`. Optional checks
(like `jupyterlab`) may show `[INFO]` if you're using VS Code -- that's
fine. Any `[FAIL]` line means the analysis won't run; the detail column
gives the fix (usually a `pip install ...` command).

**Data load shapes** (printed after `load_all`): the notebook prints the
actual row/column counts of every dataframe and the contents of
`matched_subjects`. Verify each is nonzero and that `matched_subjects`
contains the expected number of mice for the current cohort.

- `AKDdf`: every contacted/uncontacted reach in the analyzable phases.
- `FCDGdf_wide`: matched subjects x eLife groups (both/left/right hemispheres).
- `AKDdf_agg_contact`: subject x phase x contact_group aggregated kinematics.
- `matched_subjects`: mouse IDs for which both kinematic and connectomic data exist.

If any dataframe is empty or `matched_subjects` is 0, something upstream
is wrong: phase columns weren't backfilled, the wrong DB is configured,
or the imaging-parameter filter excluded everything. Re-check
`connectome.db` is the right file and that recent backfills were applied.

</details>
